# 3201 Hashing Assignment

In [1]:
import csv
import string
import time
import random
from Crypto.Hash import SHA256
from bcrypt import checkpw, hashpw
import nltk
nltk.download('words')
from nltk.corpus import words

[nltk_data] Downloading package words to
[nltk_data]     /Users/diegomelgoza/nltk_data...
[nltk_data]   Package words is already up-to-date!


## Task 1

Task 1.1

In [2]:
def hash_something(input: str):
    return SHA256.new(input.encode("utf-8")).hexdigest()

In [3]:
input = "Something"

In [4]:
print(f"Final hash: {hash_something(input)}")

Final hash: edcd3ea8b13071e3d13924cba9cc879c9101372a90706549b4d94e5337c2cbec


Task 1.2

In [5]:
def get_hamming_distance(str1:str, str2:str) -> int:
    str1_int = int.from_bytes(str1.encode("utf-8"))
    str2_int = int.from_bytes(str2.encode("utf-8"))
    and_res = str1_int ^ str2_int
    return f"{and_res:b}".count("1")

In [6]:
str1 = "cat"
str2 = "bat"
get_hamming_distance(str1, str2)

1

In [7]:
print(f"{hash_something(str1)}")
print(f"{hash_something(str2)}")

77af778b51abd4a3c51c5ddd97204a9c3ae614ebccb75a606c3b6865aed6744e
ca5bcec12f716f44d9745d349cc80422f0d14cbab09329caf533bef7c2d952eb


In [8]:
str1 = "abcdefgh"
str2 = "abbdefgh"
get_hamming_distance(str1, str2)

1

In [9]:
print(f"{hash_something(str1)}")
print(f"{hash_something(str2)}")

9c56cc51b374c3ba189210d5b6d4bf57790d351c96c47c02190ecf1e430635ab
a84ef7e28ec21592304e5a4c5578c2a018cdf96e2dea4c7d95a483287de1c501


Task 1.3

In [21]:
#create a random string using the ASCII letter set
#get each individual character by randomly selecting with Random
def random_string() -> str:
    letters = list(string.ascii_letters)
    text = ''
    for i in range(10):
        text = text + letters[random.randint(0, len(letters) - 1)]
    return text

In [22]:
#Create a new hash using encoded text
#then truncate it by
#directly extracting the needed # of bits from the bitstring
def get_truncated_hash(bits: int, text: str) -> str:
    new_hash = SHA256.new(text.encode("utf-8"))
    truncated = f"{int.from_bytes(new_hash.digest()):b}"[:bits]
    return truncated

In [32]:
#find a hash collision in a hash space "bits" # of bits
#use a dictionary to keep track of hashes as they are
#randomly generated. function with either find a collision
#or give up after the defined # of max attempts.
def find_hash_collision(bits, max_attempts):
    seen = {}
    start = time.time()
    for att in range(max_attempts):
        text = random_string()
        t_hash = get_truncated_hash(bits, text)
        if t_hash in seen:
            stop = time.time() - start
            return seen[t_hash], text, t_hash, stop, att
        else:
            seen[t_hash] = text
    stop = time.time() - start
    return None, None, None, stop, max_attempts

In [33]:
#Hash collision experiment that returns the collision values,
#the time to get a collision, and the # of hashes generated
#for every even # of bits from 8 to 50.
def hash_collision_experiment():
    results = {}
    for bits in range(8, 52, 2):
        text1, text2, hash_res, time, inputs = find_hash_collision(bits, 10000000000)
        results[bits] = [text1, text2, hash_res, time, inputs]
    with open("./hash_collision_results.txt", "w") as f:
        f.write("bits: text1, text2, hash_res, time, inputs\n")
        for bits in results:
            f.write(f"{bits}: {results[bits]}\n")


In [34]:
hash_collision_experiment()

# Part 2

In [10]:
def crack_password(pw_hash:str, word_corpus:list[str]):
    #find the word in the wordset that returns the desired hash
    for word in word_corpus:
        #checkpw from bcrypt library
        if checkpw(word.encode("utf-8"), pw_hash.encode("utf-8")):
            return word
    return None

In [12]:
def get_all_passwords(filepath:str = 'user_hashes.txt'):
    #word corpus from nltk words library
    word_corpus = [word for word in words.words() if 10 >= len(word) >= 6]
    user_hashes = {}
    user_passwords = {}
    #get every hash within the hashes file
    with open(filepath) as f:
        for line in f:
            user, hashval = line.split(":")
            print(f"{user}: {hashval}")
            user_hashes[user] = hashval

    #crack every password and write it to the temporary dictionary
    for user in user_hashes:
        password = crack_password(user_hashes[user], word_corpus)
        if password:
            user_passwords[user] = password
        else:
            user_passwords[user] = "passworderror!!!"
        print(f"Password found!: {password}")

    #write all the passwords to the file.
    with open("./user_passwords.txt", "w+") as f:
        f.write("user: password")
        for user in user_passwords:
            f.write(f"{user}: {user_passwords[user]}\n")


In [13]:
get_all_passwords()

Bilbo: $2b$08$J9FW66ZdPI2nrIMcOxFYI.qx268uZn.ajhymLP/YHaAsfBGP3Fnmq

Gandalf: $2b$08$J9FW66ZdPI2nrIMcOxFYI.q2PW6mqALUl2/uFvV9OFNPmHGNPa6YC

Thorin: $2b$08$J9FW66ZdPI2nrIMcOxFYI.6B7jUcPdnqJz4tIUwKBu8lNMs5NdT9q

Fili: $2b$09$M9xNRFBDn0pUkPKIVCSBzuwNDDNTMWlvn7lezPr8IwVUsJbys3YZm

Kili: $2b$09$M9xNRFBDn0pUkPKIVCSBzuPD2bsU1q8yZPlgSdQXIBILSMCbdE4Im

Balin: $2b$10$xGKjb94iwmlth954hEaw3O3YmtDO/mEFLIO0a0xLK1vL79LA73Gom

Dwalin: $2b$10$xGKjb94iwmlth954hEaw3OFxNMF64erUqDNj6TMMKVDcsETsKK5be

Oin: $2b$10$xGKjb94iwmlth954hEaw3OcXR2H2PRHCgo98mjS11UIrVZLKxyABK

Gloin: $2b$11$/8UByex2ktrWATZOBLZ0DuAXTQl4mWX1hfSjliCvFfGH7w1tX5/3q

Dori: $2b$11$/8UByex2ktrWATZOBLZ0Dub5AmZeqtn7kv/3NCWBrDaRCFahGYyiq

Nori: $2b$11$/8UByex2ktrWATZOBLZ0DuER3Ee1GdP6f30TVIXoEhvhQDwghaU12

Ori: $2b$12$rMeWZtAVcGHLEiDNeKCz8OiERmh0dh8AiNcf7ON3O3P0GWTABKh0O

Bifur: $2b$12$rMeWZtAVcGHLEiDNeKCz8OMoFL0k33O8Lcq33f6AznAZ/cL1LAOyK

Bofur: $2b$12$rMeWZtAVcGHLEiDNeKCz8Ose2KNe821.l2h5eLffzWoP01DlQb72O

Durin: $2b$13$6ypcazOOkUT/a7EwMuIjH.qb

KeyboardInterrupt: 